# Reproducing Figure 7: GRABACh3.0 Acetylcholine Biosensor Analysis

This notebook reproduces Figure 7 from **Belal et al. 2026** 

**Dataset**: DANDI:001832/0.260611.2102

**Analysis approach**:
- **Biosensor**: GRABACh3.0 genetically encoded acetylcholine indicator
- **Stimulation**: Single-pulse electrical stimulation to evoke acetylcholine release


## Figure 7 Analysis Procedure

This notebook reproduces the GRABACh3.0 acetylcholine biosensor analysis for Figure 7 using the downloaded NWB files from `DANDI:001832/0.260611.2102`.

The notebook first finds the repository root by searching upward for `Python functions/master_functions.py`. This means the notebook can be opened from inside `Paper analysis/Figure 7/` or from the repository root, and the helper functions should still load correctly.

All NWB input data are loaded from:

`NWB_DIR = repo / "NWBdata" / "001832"`

The main loading step is:

`df = load_all_nwb(NWB_DIR)`

`load_all_nwb(NWB_DIR)` uses the DANDI metadata file:

`NWBdata/001832/dataset_description/dataset_description.json`

to determine which NWB files belong to the figure dataset. It reads the `FigureMappings` entries, keeps mapped files that exist locally and end with `_ophys.nwb`, then loads fluorescence traces from each file's `processing["ophys"]["Fluorescence"]` module.

This builds a dataframe containing metadata and fluorescence traces for each ROI/run. The preview table shows metadata columns such as `nwb_file`, `group`, `animal_id`, `slice_id`, `roi_id`, `run_id`, `series_name`, `n_frames`, and `stim_time`.

The notebook then processes the fluorescence traces to produce averaged ROI responses. For each averaged ROI trace, the peak response is calculated as:

`fmax = max(ΔF/F)`

The data are split by condition using the `group` column:

`ctrl` = control animals

`test` = MCI-Park animals

For plotting and statistical consistency with the R analysis, the Python notebook creates a `SliceID` grouping variable. This corresponds to the random-effect grouping used in the R mixed-effects model:

`dff ~ Condition + (1 | SliceID)`

The boxplot cell compares peak GRABACh3.0 responses between control and MCI-Park groups using `boxplot_rtype`, matching the R-style boxplot behavior used elsewhere in the analysis. The boxplot uses R quantile type 6 and 1.5 × IQR whiskers, matching the custom R `BoxPlot()` helper.

The colors of the individual points indicate `SliceID`. Points from the same slice are shown in the same color. In the raw NWB data, each ROI can have repeated fluorescence runs/trials; these runs are averaged first to give one averaged response per ROI. The plotted points therefore represent averaged ROI responses, and multiple ROI-level points can come from the same `SliceID`. This matches the structure of the R analysis, where `SliceID` is treated as a random effect in the mixed-effects model. The colors are therefore not experimental groups; group identity is shown by the x-axis position (`Control` or `MCI-Park`).

The export switch is controlled by:

`SAVE = False`

If `SAVE = False`, the notebook runs the analysis and displays the plots but does not write output files.

If `SAVE = True`, summary tables are written to: `Paper analysis/Figure 7/xlsx/`

And SVG figures are written to: `Paper analysis/Figure 7/svg/`

The main exported summary table is: `Paper analysis/Figure 7/xlsx/results.xlsx`

This file contains one row per averaged ROI, with columns for animal, slice, ROI, condition, and peak `(F1-F0)/F0`.

The later example-trace cells plot individual ROI traces from `df_fl_average` using the selected row index `ii`. These cells can also export the example trace SVGs and the corresponding trace data to Excel when `SAVE = True`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

repo = Path.cwd()
while not (repo / "Python functions" / "master_functions.py").exists():
    if repo.parent == repo:
        raise FileNotFoundError("Could not find repo root containing Python functions/master_functions.py")
    repo = repo.parent

python_functions = repo / "Python functions"
if str(python_functions) not in sys.path:
    sys.path.insert(0, str(python_functions))

from master_functions import *

import warnings

warnings.filterwarnings(
    "ignore",
    message="Schema conflict.*ndx-optogenetics.*",
)


SAVE = False

EXPORT_DIR = repo / "Paper analysis" / "Figure 7" / "xlsx"
SVG_DIR = repo / "Paper analysis" / "Figure 7" / "svg"
NWB_DIR = repo / "NWBdata" / "001832"

# load file names 

dataset_json = NWB_DIR / "dataset_description" / "dataset_description.json"

with open(dataset_json, "r") as f:
    dataset_description = json.load(f)

figure7_index = {}

for entry in dataset_description["FigureMappings"]["Figure 7"]:
    nwb_path = NWB_DIR / entry["dandi_path"]

    if nwb_path.exists() and nwb_path.name.endswith("_ophys.nwb"):
        figure7_index[nwb_path] = {
            "key": entry["dandi_path"],
            "figure": "Figure 7",
            "original_path": entry.get("original_path"),
            "dandi_path": entry["dandi_path"],
        }

display(pd.DataFrame(
    [
        {
            "nwb_path": str(path),
            "dandi_path": meta["dandi_path"],
            "figure": meta["figure"],
        }
        for path, meta in figure7_index.items()
    ]
))

In [ ]:
# loads files 

records = []

for nwb_path, json_meta in figure7_index.items():
    print(f"loading {nwb_path.name}")
    records.extend(load_nwb_fluorescence(nwb_path, json_meta=json_meta))

df = pd.DataFrame(records)

if not df.empty:
    df = df.sort_values("nwb_file").reset_index(drop=True)
    df["run_id"] = (
        df.groupby(["group", "animal_id", "slice_id", "roi_id", "series_name"])
        .cumcount()
        + 1
    )

print(f"{len(df)} total records from {len(figure7_index)} files")

display(df[[
    "nwb_file",
    "group",
    "animal_id",
    "slice_id",
    "roi_id",
    "run_id",
    "series_name",
    "n_frames",
    "stim_time",
]])

## Load Figure 7 NWB Data automatically - use as an alternative

This cell sets up the Python environment, finds the repository root, loads helper functions, sets the output folders, and loads all Figure 7 NWB fluorescence data in one step.

```python
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
while not (repo / "Python functions" / "master_functions.py").exists():
    if repo.parent == repo:
        raise FileNotFoundError("Could not find repo root containing Python functions/master_functions.py")
    repo = repo.parent

python_functions = repo / "Python functions"
if str(python_functions) not in sys.path:
    sys.path.insert(0, str(python_functions))

from master_functions import *

import warnings

warnings.filterwarnings(
    "ignore",
    message="Schema conflict.*ndx-optogenetics.*",
)


SAVE = False

EXPORT_DIR = repo / "Paper analysis" / "Figure 7" / "xlsx"
SVG_DIR = repo / "Paper analysis" / "Figure 7" / "svg"
NWB_DIR = repo / "NWBdata" / "001832"

df = load_all_nwb(NWB_DIR)

# preview metadata columns (no arrays)
print(df[["nwb_file", "group", "animal_id", "slice_id", "roi_id",
          "run_id", "series_name", "n_frames", "stim_time"]])

# access one trace
row = df.iloc[0]
print("timestamps (first 5):", row["timestamps"][:5])
print("fluorescence (first 5):", row["fluorescence"][:5])
```

The repository root is found by searching upward from the current notebook location until `Python functions/master_functions.py` is found.

The main input folder is:

```python
NWB_DIR = repo / "NWBdata" / "001832"
```

The main loading command is:

```python
df = load_all_nwb(NWB_DIR)
```

`load_all_nwb(NWB_DIR)` reads:

```text
NWBdata/001832/dataset_description/dataset_description.json
```

It uses the `FigureMappings` entries to identify locally downloaded files that exist and end with `_ophys.nwb`. For each selected NWB file, it loads fluorescence traces from:

```python
processing["ophys"]["Fluorescence"]
```

The resulting dataframe `df` contains one row per fluorescence series/run, including metadata such as `nwb_file`, `group`, `animal_id`, `slice_id`, `roi_id`, `run_id`, `series_name`, `n_frames`, and `stim_time`, plus the full `timestamps` and `fluorescence` arrays.

In [ ]:
df_grab = df[df["series_name"].str.contains("GRABCh")].reset_index(drop=True)
ii = 160
trial = df_grab.iloc[ii]
t = np.asarray(trial["timestamps"], dtype=float)
f_raw = np.asarray(trial["fluorescence"], dtype=float)
stim_time = float(trial["stim_time"]) - t[0]

print(trial[["nwb_file", "group", "animal_id", "slice_id", "roi_id", "run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

pmt_background  = 144
baseline_window = 1
xlim = [4.5, 7]

dff = compute_dff(f_raw=f_raw, time=t, stim_time=stim_time, pmt_background=pmt_background, baseline_window=baseline_window)
plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, lwd=1.0, tick_len=4, axis_color="black")


In [ ]:
rows = []
for _, trial in df_grab.iterrows():
    t         = np.asarray(trial["timestamps"], dtype=float)
    f_raw     = np.asarray(trial["fluorescence"], dtype=float)
    stim_time = float(trial["stim_time"]) - t[0]
    dff       = compute_dff(f_raw=f_raw, time=t, stim_time=stim_time,
                            pmt_background=pmt_background, baseline_window=baseline_window)
    row       = trial.to_dict()
    row["fluorescence"] = dff
    rows.append(row)

df_fl = pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
ii=160
trial = df_fl.iloc[ii]

print(trial[["nwb_file","group","animal_id","slice_id","roi_id","run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

t = np.asarray(trial["timestamps"], dtype=float)           
dff = np.asarray(trial["fluorescence"], dtype=float)
       
plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, lwd=1.0, tick_len=4, axis_color="black")

In [ ]:
group_cols = ["group", "animal_id", "slice_id", "roi_id"]
rows = []
for keys, grp in df_fl.groupby(group_cols, sort=False):
    traces = [np.asarray(f, dtype=float) for f in grp["fluorescence"].values]
    min_len = min(len(t) for t in traces)
    traces_clipped = np.vstack([t[:min_len] for t in traces])
    dff_avg = traces_clipped.mean(axis=0)
    ref = grp.sort_values("run_id").iloc[0].to_dict()
    ref["fluorescence"] = dff_avg
    ref["timestamps"]   = np.asarray(ref["timestamps"], dtype=float)[:min_len]
    ref["n_reps"]       = len(grp)
    rows.append(ref)
df_fl_average = pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
df_fl_average

In [ ]:
# find peak ΔF/F for each averaged ROI
df_fl_average["fmax"] = df_fl_average["fluorescence"].apply(
    lambda f: np.asarray(f, dtype=float).max()
)

# split by condition
ctrl = df_fl_average[df_fl_average["group"] == "ctrl"]
test = df_fl_average[df_fl_average["group"] == "test"]

print("ctrl fmax:", ctrl["fmax"].describe())
print("test fmax:", test["fmax"].describe())


In [ ]:
# Figure 7 boxplot styled to match R BoxPlot()

plt.close("all")
%matplotlib inline

from matplotlib.colors import LinearSegmentedColormap, to_rgba

save_plot = SAVE

axis_color = "black"
lwd = 4 / 3
wid = 0.3
cap = 0.05
amount = 0.05
p_cex = 0.6

# R BoxPlot() uses colorRampPalette(viridis stops)(n), alpha=0.6
viridis_stops = ["#440154", "#3B528B", "#21918C", "#5DC963", "#FDE725"]
slice_cmap = LinearSegmentedColormap.from_list("r_boxplot_viridis", viridis_stops)

plot_df = df_fl_average.copy()

# Match the R random-effect grouping: dff ~ Condition + (1 | SliceID)
plot_df["Condition"] = plot_df["group"].replace({
    "ctrl": "Control",
    "test": "MCI-Park",
})

plot_df["SliceID"] = (
    plot_df["group"].astype(str)
    + "_animal" + plot_df["animal_id"].astype(str)
    + "_slice" + plot_df["slice_id"].astype(str)
)

ctrl = plot_df[plot_df["Condition"] == "Control"]
test = plot_df[plot_df["Condition"] == "MCI-Park"]

slice_ids = pd.unique(plot_df["SliceID"])
slice_colors = {
    slice_id: to_rgba(slice_cmap(i / max(len(slice_ids) - 1, 1)), alpha=0.6)
    for i, slice_id in enumerate(slice_ids)
}

fig, ax = plt.subplots(figsize=(3, 3.5))
fig.patch.set_alpha(0)
ax.set_facecolor("none")

box_plot = boxplot_rtype(
    ax,
    [ctrl["fmax"].values, test["fmax"].values],
    rtype=6,
    whis=1.5,
    positions=[1, 2],
    patch_artist=True,
    widths=wid,
    showfliers=False,
    boxprops=dict(edgecolor=axis_color, linewidth=lwd),
    whiskerprops=dict(color=axis_color, linewidth=lwd),
    capprops=dict(color=axis_color, linewidth=lwd),
    median_overhang=0.03,
    medianprops=dict(color=axis_color, linewidth=3 * lwd),
    showpoints=False,
    paired=False,
)

for box in box_plot["boxes"]:
    box.set_facecolor("white")
    box.set_edgecolor(axis_color)

rng = np.random.default_rng(42)
point_size = (p_cex * 10) ** 2

for xpos, condition in zip([1, 2], ["Control", "MCI-Park"]):
    group_df = plot_df[plot_df["Condition"] == condition].copy()

    x = xpos + rng.uniform(-amount, amount, size=len(group_df))
    colors = [slice_colors[slice_id] for slice_id in group_df["SliceID"]]

    ax.scatter(
        x,
        group_df["fmax"],
        s=point_size,
        c=colors,
        edgecolors="none",
        zorder=3,
    )

ax.set_xticks([1, 2])
ax.set_xticklabels(["Control", "MCI-Park"], rotation=45, ha="right")
ax.set_ylabel("peak ΔF/F")
ax.set_xlim(0.75, 2.25)
ax.set_ylim(0, 1.5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(axis_color)
ax.spines["bottom"].set_color(axis_color)
ax.spines["left"].set_linewidth(lwd)
ax.spines["bottom"].set_linewidth(lwd)
ax.tick_params(direction="out", length=4, width=lwd, color=axis_color, labelcolor=axis_color)

plt.tight_layout()

if save_plot:
    SVG_DIR.mkdir(parents=True, exist_ok=True)
    boxplot_svg_path = SVG_DIR / "fmax_boxplot.svg"
    fig.savefig(boxplot_svg_path, format="svg", bbox_inches="tight", transparent=True)
    print(f"Exported: {boxplot_svg_path}")

plt.show()


In [ ]:
ctrl_ids = sorted(df_fl_average[df_fl_average['group'] == 'ctrl']['animal_id'].unique())
test_ids = sorted(df_fl_average[df_fl_average['group'] != 'ctrl']['animal_id'].unique())

n_ctrl = len(ctrl_ids)

# Relabel test animals after ctrl animals
test_relabel = {orig: n_ctrl + i + 1 for i, orig in enumerate(test_ids)}

df_export = df_fl_average.copy()

df_export['Animal'] = df_export.apply(
    lambda r: r['animal_id'] if r['group'] == 'ctrl'
    else test_relabel[r['animal_id']],
    axis=1
)

# Rename condition values
df_export['group'] = df_export['group'].replace({
    'ctrl': 'Control',
    'test': 'MCI-Park'
})

out = (
    df_export.rename(columns={
        'slice_id': 'Slice',
        'roi_id': 'ROI',
        'group': 'Condition',
        'fmax': '(F1-F0)/F0'
    })
    [['Animal', 'Slice', 'ROI', 'Condition', '(F1-F0)/F0']]
    .sort_values(['Animal', 'Slice', 'ROI'])
    .reset_index(drop=True)
)

if SAVE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    out.to_excel(EXPORT_DIR / "results.xlsx", index=False)
    print(f"Exported: {EXPORT_DIR / 'results.xlsx'}")

In [ ]:
# single examples
%matplotlib widget

ii=8
trial = df_fl_average.iloc[ii]
xlim = [4.5, 7]
ylim = [-0.05, 0.8]

vmin = 0
vmax = 0.8
print(trial[["nwb_file","group","animal_id","slice_id","roi_id","run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

t = np.asarray(trial["timestamps"], dtype=float)           
dff = np.asarray(trial["fluorescence"], dtype=float)
       
trace_svg_path = None
if SAVE:
    SVG_DIR.mkdir(parents=True, exist_ok=True)
    trace_svg_path = SVG_DIR / f"single_example_trace_ii{ii}_animal{trial['animal_id']}_slice{trial['slice_id']}_roi{trial['roi_id']}.svg"

plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, ylim=ylim, lwd=1.0, tick_len=4, vmin=vmin, vmax=vmax, axis_color="black", savepath=trace_svg_path)
if SAVE:
    print(f"Exported: {trace_svg_path}")

# export single eg
cond = {"ctrl": "Control", "test": "MCI-Park"}.get(trial["group"], trial["group"])

meta_out = pd.DataFrame([{
    "Animal": trial["animal_id"],
    "Slice": trial["slice_id"],
    "ROI": trial["roi_id"],
    "Condition": cond,
    "run_id": trial["run_id"],
    "nwb_file": trial["nwb_file"],
    "stim_time_s": stim_time,
    "(F1-F0)/F0_peak": float(np.nanmax(dff)),
}])

trace_out = pd.DataFrame({
    "time_s": t,
    "time_from_start_s": t - t[0],
    "dff": dff,
})

if SAVE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    export_path = EXPORT_DIR / f"single_example_animal{trial['animal_id']}_slice{trial['slice_id']}_roi{trial['roi_id']}.xlsx"
    with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
        meta_out.to_excel(writer, sheet_name="summary", index=False)
        trace_out.to_excel(writer, sheet_name="trace", index=False)

    print(f"Exported: {export_path}")


In [ ]:
ii=30
trial = df_fl_average.iloc[ii]

print(trial[["nwb_file","group","animal_id","slice_id","roi_id", "run_id"]])
print(f"stim_time={stim_time:.2f}s  n_frames={len(t)}")

t = np.asarray(trial["timestamps"], dtype=float)           
dff = np.asarray(trial["fluorescence"], dtype=float)
       
trace_svg_path = None
if SAVE:
    SVG_DIR.mkdir(parents=True, exist_ok=True)
    trace_svg_path = SVG_DIR / f"single_example_trace_ii{ii}_animal{trial['animal_id']}_slice{trial['slice_id']}_roi{trial['roi_id']}.svg"

plot_dff(time=t, dff=dff, stim_time=stim_time, xlim=xlim, ylim=ylim, lwd=1.0, tick_len=4, vmin=vmin, vmax=vmax, axis_color="black", savepath=trace_svg_path)
if SAVE:
    print(f"Exported: {trace_svg_path}")

# export single eg
cond = {"ctrl": "Control", "test": "MCI-Park"}.get(trial["group"], trial["group"])

meta_out = pd.DataFrame([{
    "Animal": trial["animal_id"],
    "Slice": trial["slice_id"],
    "ROI": trial["roi_id"],
    "Condition": cond,
    "run_id": trial["run_id"],
    "nwb_file": trial["nwb_file"],
    "stim_time_s": stim_time,
    "(F1-F0)/F0_peak": float(np.nanmax(dff)),
}])

trace_out = pd.DataFrame({
    "time_s": t,
    "time_from_start_s": t - t[0],
    "dff": dff,
})

if SAVE:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    export_path = EXPORT_DIR / f"single_example_animal{trial['animal_id']}_slice{trial['slice_id']}_roi{trial['roi_id']}.xlsx"
    with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
        meta_out.to_excel(writer, sheet_name="summary", index=False)
        trace_out.to_excel(writer, sheet_name="trace", index=False)

    print(f"Exported: {export_path}")